## SPRINT-2 BREIF-2 BANK TRANSACTIONS

## Connection

In [3]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"

engine = create_engine(DATABASE_URL)

## Create tables

In [4]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS dim_clients(
            client_id VARCHAR PRIMARY KEY NOT NULL,
            score_credit_client DECIMAL,
            segment_client VARCHAR,
            categorie_risque VARCHAR 
        );

        CREATE TABLE IF NOT EXISTS dim_produits(
            produit_id SERIAL PRIMARY KEY,
            produit VARCHAR UNIQUE
        );

        CREATE TABLE IF NOT EXISTS dim_agences(
            agence_id SERIAL PRIMARY KEY,
            agence VARCHAR UNIQUE
        );

        CREATE TABLE IF NOT EXISTS dim_categories(
            categorie_id SERIAL PRIMARY KEY,
            categorie VARCHAR UNIQUE
        );

        CREATE TABLE IF NOT EXISTS dim_dates(
            date_id SERIAL PRIMARY KEY,
            anne INT,
            trimestre INT,
            mois INT,
            jour_semaine VARCHAR
        );

        CREATE TABLE IF NOT EXISTS fact_transactions(
            transaction_id VARCHAR PRIMARY KEY,

            client_id VARCHAR,
            produit_id INT,
            agence_id INT,
            categorie_id INT,
            date_id INT,

            montant DECIMAL,
            taux_change_eur DECIMAL,
            montant_eur DECIMAL,

            type_operation VARCHAR,
            statut VARCHAR,

            crédits DECIMAL,
            débits DECIMAL,
            solde_net DECIMAL,

            is_anomaly_montant BOOLEAN,
            is_conversion_error BOOLEAN,

            FOREIGN KEY (client_id) REFERENCES dim_clients(client_id),
            FOREIGN KEY (produit_id) REFERENCES dim_produits(produit_id),
            FOREIGN KEY (agence_id) REFERENCES dim_agences(agence_id),
            FOREIGN KEY (categorie_id) REFERENCES dim_categories(categorie_id),
            FOREIGN KEY (date_id) REFERENCES dim_dates(date_id)
        );
'''))

## Create Indexs

In [5]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE INDEX idx_fact_client ON fact_transactions(client_id);
        CREATE INDEX idx_fact_date ON fact_transactions(date_id);
        CREATE INDEX idx_fact_agence ON fact_transactions(agence_id);
'''))

## Data Loading

In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\HP\Desktop\financecore_project\data\financecore_clean.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1925 entries, 0 to 1924
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       1925 non-null   str    
 1   client_id            1925 non-null   str    
 2   date_transaction     1925 non-null   str    
 3   montant              1925 non-null   float64
 4   devise               1925 non-null   str    
 5   taux_change_eur      1925 non-null   float64
 6   montant_eur          1925 non-null   float64
 7   categorie            1925 non-null   str    
 8   produit              1925 non-null   str    
 9   agence               1925 non-null   str    
 10  type_operation       1925 non-null   str    
 11  statut               1925 non-null   str    
 12  score_credit_client  1925 non-null   float64
 13  segment_client       1925 non-null   str    
 14  solde_avant          1925 non-null   float64
 15  is_anomaly_montant   1925 non-null   bool   
 16 

In [7]:
dim_clients_df = df[["client_id", "score_credit_client", "segment_client", "categorie_risque"]].drop_duplicates(subset=["client_id"])
dim_produits_df = df[["produit"]].drop_duplicates()
dim_agences_df = df[["agence"]].drop_duplicates()
dim_categories_df = df[["categorie"]].drop_duplicates()
dim_dates_df = df[["anne", "trimestre", "mois", "jour_semaine"]].drop_duplicates()

In [8]:
dim_clients_df.to_sql(
    "stg_clients",
    engine,
    if_exists="replace",
    index=False
)

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO dim_clients (
            client_id,
            score_credit_client,
            segment_client,
            categorie_risque
        )
        SELECT DISTINCT
            client_id,
            score_credit_client,
            segment_client,
            categorie_risque
        FROM stg_clients
        WHERE client_id IS NOT NULL

        ON CONFLICT (client_id)
        DO UPDATE SET
            score_credit_client = EXCLUDED.score_credit_client,
            segment_client = EXCLUDED.segment_client,
            categorie_risque = EXCLUDED.categorie_risque;
    """))

In [9]:
dim_produits_df.to_sql(
    "stg_produits",
    engine,
    if_exists="replace",
    index=False
)

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO dim_produits (produit)
        SELECT DISTINCT produit
        FROM stg_produits
        WHERE produit IS NOT NULL
        ON CONFLICT (produit) DO NOTHING;
    """))

dim_produits_db = pd.read_sql(
    "SELECT * FROM dim_produits",
    engine
)

df = df.merge(
    dim_produits_db,
    on="produit",
    how="left"
)

In [10]:
dim_agences_df.to_sql(
    "stg_agences",
    engine,
    if_exists="replace",
    index=False
)

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO dim_agences (agence)
        SELECT DISTINCT agence
        FROM stg_agences
        WHERE agence IS NOT NULL
        ON CONFLICT (agence) DO NOTHING;
    """))

dim_agences_db = pd.read_sql(
    "SELECT * FROM dim_agences",
    engine
)

df = df.merge(
    dim_agences_db,
    on="agence",
    how="left"
)

In [11]:
dim_categories_df.to_sql(
    "stg_categories",
    engine,
    if_exists="replace",
    index=False
)

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO dim_categories (categorie)
        SELECT DISTINCT categorie
        FROM stg_categories
        WHERE categorie IS NOT NULL
        ON CONFLICT (categorie) DO NOTHING;
    """))

dim_categories_db = pd.read_sql(
    "SELECT * FROM dim_categories",
    engine
)

df = df.merge(
    dim_categories_db,
    on="categorie",
    how="left"
)

In [12]:
dim_dates_df.to_sql(
    "dim_dates",
    engine,
    if_exists="append",
    index=False
)

dim_dates_db = pd.read_sql(
    "SELECT * FROM dim_dates",
    engine
)

df = df.merge(
    dim_dates_db,
    on=["anne", "trimestre", "mois", "jour_semaine"],
    how="left"
)

In [13]:
fact_transactions_df = df[[
    "transaction_id",
    "client_id",
    "produit_id",
    "agence_id",
    "categorie_id",
    "date_id",
    "montant",
    "taux_change_eur",
    "montant_eur",
    "type_operation",
    "statut",
    "crédits",
    "débits",
    "solde_net",
    "is_anomaly_montant",
    "is_conversion_error"
]]

In [14]:
fact_transactions_df.to_sql(
    "fact_transactions",
    engine,
    if_exists="append",
    index=False
)

775

## Check

In [15]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT 
                SUM(CASE WHEN produit_id IS NULL THEN 1 ELSE 0 END) AS null_produits,
                SUM(CASE WHEN agence_id IS NULL THEN 1 ELSE 0 END) AS null_agences,
                SUM(CASE WHEN categorie_id IS NULL THEN 1 ELSE 0 END) AS null_categories, 
                SUM(CASE WHEN date_id IS NULL THEN 1 ELSE 0 END) AS null_dates
        FROM fact_transactions;
'''))
    
print(result.fetchone())

(0, 0, 0, 0)


## SQL Analytics

In [16]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
		        a.agence,
		        p.produit,
		        d.mois,
		        COUNT(t.transaction_id) AS total_transactions,
		        ROUND(AVG(COUNT(t.transaction_id)) OVER()) AS average_transactions
        FROM fact_transactions t
        JOIN dim_agences a
		        ON t.agence_id = a.agence_id
        JOIN dim_produits p 
		        ON t.produit_id = p.produit_id
        JOIN dim_dates d 
		        ON t.date_id = d.date_id
        GROUP BY a.agence, p.produit, d.mois
        ORDER BY total_transactions DESC
        LIMIT 10;
'''))
    
print(result)

In [17]:
with engine.begin() as conn:
    result = conn.execute(text('''
SELECT 
	c.client_id,
	t.solde_net
FROM fact_transactions t
JOIN dim_clients c
	ON t.client_id = c.client_id
WHERE solde_net < (SELECT ROUND(AVG(t.solde_net)::numeric, 2) FROM fact_transactions t)
ORDER BY solde_net DESC
LIMIT 10;
'''))
    
print(result)